# CNN Practice for Timeseries
Heavily inspired by Mayur Ingole, https://medium.com/@myringoleMLGOD/simple-convolutional-neural-network-cnn-for-dummies-in-pytorch-a-step-by-step-guide-6f4109f6df80

In [20]:
# Use vnv timeseries_cnn_test with Python 3.13.9

import torch
import numpy as np
from torch.utils.data import Dataset
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

import torch.nn.functional as F
import torchvision.datasets as datasets
from torch import optim
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import scipy
import re
from pathlib import Path
import pandas as pd

torch.manual_seed(42)

In [5]:

def imshow(sample_element, shape = (28, 28)):
    plt.imshow(sample_element[0].numpy().reshape(shape), cmap='gray')
    plt.title('Label = ' + str(sample_element[1]))
    plt.show()

In [6]:
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes=10):
        """
        Define the layers of the convolutional neural network.

        Parameters:
            in_channels: int
                The number of channels in the input image. For MNIST, this is 1 (grayscale images).
            num_classes: int
                The number of classes we want to predict, in our case 10 (digits 0 to 9).
        """
        super(CNN, self).__init__()

        # First convolutional layer: 1 input channel, 8 output channels, 3x3 kernel, stride 1, padding 1
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=8, kernel_size=3, stride=1, padding=1)
        # Max pooling layer: 2x2 window, stride 2
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Second convolutional layer: 8 input channels, 16 output channels, 3x3 kernel, stride 1, padding 1
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=1, padding=1)
        # Fully connected layer: 16*7*7 input features (after two 2x2 poolings), 10 output features (num_classes)
        self.fc1 = nn.Linear(16 * 7 * 7, num_classes)

    def forward(self, x):
        """
        Define the forward pass of the neural network.

        Parameters:
            x: torch.Tensor
                The input tensor.

        Returns:
            torch.Tensor
                The output tensor after passing through the network.
        """
        x = F.relu(self.conv1(x))  # Apply first convolution and ReLU activation
        x = self.pool(x)           # Apply max pooling
        x = F.relu(self.conv2(x))  # Apply second convolution and ReLU activation
        x = self.pool(x)           # Apply max pooling
        x = x.reshape(x.shape[0], -1)  # Flatten the tensor
        x = self.fc1(x)            # Apply fully connected layer
        return x

In [7]:
device = "mps" if torch.cuda.is_available() else "cpu"

In [ ]:
### Hyperparameters ###

input_size = 784  # 28x28 pixels (not directly used in CNN)
num_classes = 10  # digits 0-9
learning_rate = 0.001
batch_size = 64
num_epochs = 10  # Reduced for demonstration purposes

In [ ]:
### Pre-Process the Data ### 


fs = 128.0 # DATA RECORDED AT 128HZ for GSR and ECG as well.



# Load self-reported annotations
annotations_file = "/Users/Ben/Desktop/ECG_GSR_Emotions/Self-Annotation Labels/Self-annotation Multimodal_Use.xlsx"
annotations = pd.read_excel(annotations_file)
#print(annotations.head)

all_samples = pd.DataFrame({})

# TODO: Turn into a for loop
path = "/Users/Ben/Desktop/ECG_GSR_Emotions/Raw Data/Multimodal/ECG/ECGdata_s1p1v1.dat"
import numpy as np
x = np.loadtxt(path, delimiter=",")   # add delimiter="," for commas, skiprows=1 for a header
print(x.shape)

t = np.arange(len(x)) / fs

window_size = 5.0 # In seconds
num_elems_in_window = int(window_size * (fs / 60.0))

# Take the raw ECG values and split into windows
windows = np.lib.stride_tricks.sliding_window_view(x, num_elems_in_window)
print(num_elems_in_window)

# Look up the annotations for each file
def parse_spv(path):
    m = re.search(r"s(\d+)p(\d+)v(\d+)$", Path(path).stem)
    if not m:
        raise ValueError(f"No s/p/v pattern at the end of {path}")
    return tuple(int(g) for g in m.groups())

s, p, v = parse_spv(path)
print(f"s={s}, p={p}, v={v}")

row = annotations[(annotations["Participant Id"] == s) & (annotations["Session ID"] == p) & (annotations["Video ID"] == v)]
row = row.drop(columns=["Name", "Age", "Gender"])

for w in windows:
    # Add window to training set with values from the annotations
    row_copy = row.copy()
    row_copy.insert(0, "window_sample", [w])

    all_samples = pd.concat([all_samples, row_copy])

print(all_samples)

(5000,)
10
s=1, p=1, v=1
     window_sample  Participant Id  Session ID  Video ID  Valence level  \
0  [1, 1, 1, 1, 1]               1           1         1              8   

   Arousal level  Dominance level Happy      Sad     Fear    Anger  Neutral  \
0              0                0  High  VeryLow  VeryLow  VeryLow  VeryLow   

   Disgust Surprised Familiarity Score Emotion V_Label A_Label Four_Labels  
0  VeryLow   VeryLow     Never watched   Happy      HV      LA        HVLA  


In [ ]:
### Load the data ###
# train_set = DataLoader()

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 11)

# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)

# classes = dataset.classes
# print(classes)

train_dataset = datasets.MNIST(root="dataset/", download=True, train=True, transform=transforms.ToTensor())
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = datasets.MNIST(root="dataset/", download=True, train=False, transform=transforms.ToTensor())
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)




# Split the data into traing and test (no validation for now)

In [ ]:
### Init Model ###

model = CNN(in_channels=1, num_classes=num_classes).to(device)

In [ ]:
# TRAIN 🚂🚂🚂

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    print(f"Epoch [{epoch + 1}/{num_epochs}]")
    for batch_index, (data, targets) in enumerate(tqdm(train_loader)):
        # Move data and targets to the device (GPU/CPU)
        data = data.to(device)
        targets = targets.to(device)

        # Forward pass: compute the model output
        scores = model(data)
        loss = criterion(scores, targets)

        # Backward pass: compute the gradients
        optimizer.zero_grad()
        loss.backward()

        # Optimization step: update the model parameters
        optimizer.step()

model.train()
optimizer.zero_grad()
predictions = model(batch)
loss = criterion(predictions, batch_2)
loss.backward()
optimizer.step()

In [ ]:
### Check Accuracy ###

def check_accuracy(loader, model):
    """
    Checks the accuracy of the model on the given dataset loader.

    Parameters:
        loader: DataLoader
            The DataLoader for the dataset to check accuracy on.
        model: nn.Module
            The neural network model.
    """
    if loader.dataset.train:
        print("Checking accuracy on training data")
    else:
        print("Checking accuracy on test data")

    num_correct = 0
    num_samples = 0
    model.eval()  # Set the model to evaluation mode

    with torch.no_grad():  # Disable gradient calculation
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            # Forward pass: compute the model output
            scores = model(x)
            _, predictions = scores.max(1)  # Get the index of the max log-probability
            num_correct += (predictions == y).sum()  # Count correct predictions
            num_samples += predictions.size(0)  # Count total samples

        # Calculate accuracy
        accuracy = float(num_correct) / float(num_samples) * 100
        print(f"Got {num_correct}/{num_samples} with accuracy {accuracy:.2f}%")
    
    model.train()  # Set the model back to training mode

# Final accuracy check on training and test sets
check_accuracy(train_loader, model)
check_accuracy(test_loader, model)